### Análisis de canasta de mercado


#### Limpieza y preparación de datos

In [ ]:
import pandas as pd
order_details= pd.read_csv('order_details.csv')
order_details.head(3)

In [2]:
orders= pd.read_csv('orders.csv')
orders.head(3)

,order_id,date,time
0,1,2015-01-01,11:38:36
1,2,2015-01-01,11:57:40
2,3,2015-01-01,12:12:28


In [3]:
pizza_types= pd.read_csv('pizza_types.csv', encoding= 'latin')
pizza_types.head(3)

,pizza_type_id,name,category,ingredients
0,bbq_ckn,The Barbecue Chicken Pizza,Chicken,"Barbecued Chicken, Red Peppers, Green Peppers,..."
1,cali_ckn,The California Chicken Pizza,Chicken,"Chicken, Artichoke, Spinach, Garlic, Jalapeno ..."
2,ckn_alfredo,The Chicken Alfredo Pizza,Chicken,"Chicken, Red Onions, Red Peppers, Mushrooms, A..."


In [4]:
pizzas= pd.read_csv('pizzas.csv')
pizzas.head(3)

,pizza_id,pizza_type_id,size,price
0,bbq_ckn_s,bbq_ckn,S,12.75
1,bbq_ckn_m,bbq_ckn,M,16.75
2,bbq_ckn_l,bbq_ckn,L,20.75


In [5]:
# Unión de dataframes por órden
orders_df= orders.merge(order_details, on= 'order_id', how= 'left')

In [6]:
# Unión por tipos de pizza
pizza_df= orders_df.merge(pizzas, on= 'pizza_id', how= 'left')
df= pizza_df.merge(pizza_types, on= 'pizza_type_id', how= 'left')

In [7]:
# Las longitudes de ambos dataframes coinciden
print(order_details.__len__() == df.__len__())

True


In [39]:
# Dataframe conformado
df.head(3)

,order_id,date,time,order_details_id,pizza_id,quantity,pizza_type_id,size,price,name,category,ingredients
0,1,2015-01-01,11:38:36,1,hawaiian_m,1,hawaiian,M,13.25,The Hawaiian Pizza,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese"
1,2,2015-01-01,11:57:40,2,classic_dlx_m,1,classic_dlx,M,16.00,The Classic Deluxe Pizza,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,..."
2,2,2015-01-01,11:57:40,3,five_cheese_l,1,five_cheese,L,18.50,The Five Cheese Pizza,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go..."


In [26]:
# Búsqueda de valores duplicados
print(df.duplicated().sum())

0


In [27]:
# Búsqueda de nulos y tipos de datos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          48620 non-null  int64  
 1   date              48620 non-null  object 
 2   time              48620 non-null  object 
 3   order_details_id  48620 non-null  int64  
 4   pizza_id          48620 non-null  object 
 5   quantity          48620 non-null  int64  
 6   pizza_type_id     48620 non-null  object 
 7   size              48620 non-null  object 
 8   price             48620 non-null  float64
 9   name              48620 non-null  object 
 10  category          48620 non-null  object 
 11  ingredients       48620 non-null  object 
dtypes: float64(1), int64(3), object(8)
memory usage: 4.5+ MB


In [7]:
# Conversión de datos fecha y categoría
df['date']= pd.to_datetime(df['date'], yearfirst= True)

In [8]:
df['name']= df['name'].astype('category')

In [9]:
# Conversión de datos de tipo objeto
cols= df.select_dtypes(include= 'object').columns
for col in df[cols]:
    df[col]= df[col].astype('string')
df.dtypes

order_id                     int64
date                datetime64[ns]
time                string[python]
order_details_id             int64
pizza_id            string[python]
quantity                     int64
pizza_type_id       string[python]
size                string[python]
price                      float64
name                      category
category            string[python]
ingredients         string[python]
dtype: object

#### Conformación de listas 

In [10]:
expanded= df.loc[df.index.repeat(df['quantity'])]

In [11]:
# Obtenemos listas con los productos adquiridos por cada transacción u órden
# Facilita la aplicación del algoritmo Apriori
grouped= expanded.groupby('order_id')['name'].apply(list)
grouped

order_id
1                                     [The Hawaiian Pizza]
2        [The Classic Deluxe Pizza, The Five Cheese Piz...
3        [The Italian Supreme Pizza, The Prosciutto and...
4                              [The Italian Supreme Pizza]
5                              [The Italian Supreme Pizza]
                               ...                        
21346    [The Big Meat Pizza, The California Chicken Pi...
21347    [The Barbecue Chicken Pizza, The Italian Supre...
21348    [The Chicken Alfredo Pizza, The Four Cheese Pi...
21349                                 [The Mexicana Pizza]
21350                         [The Barbecue Chicken Pizza]
Name: name, Length: 21350, dtype: object

In [12]:
import itertools
pairs= []
for i in grouped:
    pairs.extend(list(itertools.combinations(i, 2)))

In [13]:
# Primeros 10 pares
pairs[0:10]

[('The Classic Deluxe Pizza', 'The Five Cheese Pizza'),
 ('The Classic Deluxe Pizza', 'The Italian Supreme Pizza'),
 ('The Classic Deluxe Pizza', 'The Mexicana Pizza'),
 ('The Classic Deluxe Pizza', 'The Thai Chicken Pizza'),
 ('The Five Cheese Pizza', 'The Italian Supreme Pizza'),
 ('The Five Cheese Pizza', 'The Mexicana Pizza'),
 ('The Five Cheese Pizza', 'The Thai Chicken Pizza'),
 ('The Italian Supreme Pizza', 'The Mexicana Pizza'),
 ('The Italian Supreme Pizza', 'The Thai Chicken Pizza'),
 ('The Mexicana Pizza', 'The Thai Chicken Pizza')]

#### Reglas de asociación

Se construyen reglas de asociación para identificar productos comprados juntos con frecuencia:<br> 
<center>'Si un cliente compra A, es probable que también compre B' <b>(antecedente -> consecuente).</b> </center><br>
Teniendo esto en cuenta, una regla de asociación pueden ser <b>multi antecedente</b>, ej. {pan, galletas} -> {leche} o <b>multi consecuente</b>, ej. {hamburguesa} -> {refresco, papas}


In [14]:
from mlxtend.preprocessing import TransactionEncoder

In [15]:
# Aplicación de fit y transform en el objeto encoder
encoder= TransactionEncoder().fit(pairs)
onehot= encoder.transform(pairs)

In [16]:
# Conversión a formato one-hot (matriz binaria)
df_onehot= pd.DataFrame(onehot, columns= encoder.columns_)
df_onehot.head()

,The Barbecue Chicken Pizza,The Big Meat Pizza,The Brie Carre Pizza,The Calabrese Pizza,The California Chicken Pizza,The Chicken Alfredo Pizza,The Chicken Pesto Pizza,The Classic Deluxe Pizza,The Five Cheese Pizza,The Four Cheese Pizza,...,The Prosciutto and Arugula Pizza,The Sicilian Pizza,The Soppressata Pizza,The Southwest Chicken Pizza,The Spicy Italian Pizza,The Spinach Pesto Pizza,The Spinach Supreme Pizza,The Spinach and Feta Pizza,The Thai Chicken Pizza,The Vegetables + Vegetables Pizza
0,False,False,False,False,False,False,False,True,True,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False


In [188]:
# Pizzas más compradas en promedio
df_onehot.mean().sort_values(ascending= False).head()

The Pepperoni Pizza           0.096417
The Hawaiian Pizza            0.095554
The Barbecue Chicken Pizza    0.095451
The Thai Chicken Pizza        0.093988
The Classic Deluxe Pizza      0.093725
dtype: float64

#### Algoritmo Apriori<br>
Es uno de los algoritmos más utilizados en la minería de reglas de asociación. El algoritmo identifica elementos que aparecen juntos con frecuencia para generar las reglas de asociación. En la poda (pruning) se eliminan conjuntos que no alcanzan un umbral mínimo de soporte, entre otros factores.
<br><br> <b> Al decidir qué reglas de asociación son convenientes deben evaluarse los valores de varias métricas antes de llegar a una conclusión</b>.<br><br>En la lista siguiente, <b>los valores mínimos recomendados dependen del tamaño del set de datos</b>.
<br><br>
- <b>Soporte (support)</b>: Frecuencia de un conjunto de elementos en el dataset. Soporte mínimo: de 5-10%
- <b>Confianza (confidence)</b>: Probabilidad de que en una transacción aparezca un elemento (consecuente), dado que otro está presente (antecedente). Confianza mínima: 60-70%
- <b>Elevación (Lift)</b>: Indica qué tan intensa es la asociación, si los elementos se compran juntos con más frecuencia de lo esperado. Lift mínimo: 1.0-1.5
<br><br>Otros parámetros de complemento:<br><br>
- <b>Antecedent support</b>: Frecuencia del antecedente en el dataset, recomendado: 10%
- <b>Leverage</b>: Identifica reglas con información más allá de azar.
- <b>Conviction</b>: Mide la fuerza de la implicación en las reglas.
- <b>Zhang's metric</b>:  Métrica balanceada para asociaciones positivas o negativas.

In [17]:
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
# Se realiza la poda de conjuntos frecuentes con soporte mínimo del 1% 
frequent_itemsets= apriori(df_onehot, min_support= 0.01, max_len= None, use_colnames= True)
print('Cantidad de reglas generadas:' , len(frequent_itemsets))

Cantidad de reglas generadas: 32


In [ ]:
# Extrae todas las reglas de asociación con confianza mínima del 60%
positive_rules= association_rules(frequent_itemsets, metric= 'confidence', min_threshold= 0.6)
print('Cantidad de reglas generadas:', len(positive_rules))

Cantidad de reglas generadas: 0


In [171]:
# Ajuste de poda a un número menor 
frequent_itemsets= apriori(df_onehot, min_support= 0.005, max_len= None, use_colnames= True)
print('Cantidad de reglas generadas:', len(frequent_itemsets))

Cantidad de reglas generadas: 33


In [ ]:
# Extrae todas las reglas con confianza mínima del 5%
positive_rules= association_rules(frequent_itemsets, metric= 'confidence', min_threshold= 0.05)
print('Cantidad de reglas generadas:', len(positive_rules))

Cantidad de reglas generadas: 2


In [72]:
positive_rules.columns

Index(['antecedents', 'consequents', 'antecedent support',
       'consequent support', 'support', 'confidence', 'lift',
       'representativity', 'leverage', 'conviction', 'zhangs_metric',
       'jaccard', 'certainty', 'kulczynski'],
      dtype='object')

In [175]:
positive_rules[['antecedents', 'consequents', 'antecedent support', 'support', 'confidence', 'lift', 'conviction', 'zhangs_metric']]

,antecedents,consequents,antecedent support,support,confidence,lift,conviction,zhangs_metric
0,(The Hawaiian Pizza),(The Thai Chicken Pizza),0.095554,0.005194,0.054356,0.578329,0.958090,-0.446336
1,(The Thai Chicken Pizza),(The Hawaiian Pizza),0.093988,0.005194,0.055262,0.578329,0.957351,-0.445908


In [76]:
# Importamos segundo set de ejemplo convertido a matriz binaria
df_market= pd.read_csv('market.csv', sep=';')
df_market.head(3)

,Bread,Honey,Bacon,Toothpaste,Banana,Apple,Hazelnut,Cheese,Meat,Carrot,...,Milk,Butter,ShavingFoam,Salt,Flour,HeavyCream,Egg,Olive,Shampoo,Sugar
0,1,0,1,0,1,1,1,0,0,1,...,0,0,0,0,0,1,1,0,0,1
1,1,1,1,0,1,1,1,0,0,0,...,1,1,0,0,1,0,0,1,1,0
2,0,1,1,1,1,1,1,1,1,0,...,1,0,1,1,1,1,1,0,0,1


In [ ]:
# Conversión de datos 
df_market= df_market.astype('bool')

In [260]:
df_market.dtypes

Bread          bool
Honey          bool
Bacon          bool
Toothpaste     bool
Banana         bool
Apple          bool
Hazelnut       bool
Cheese         bool
Meat           bool
Carrot         bool
Cucumber       bool
Onion          bool
Milk           bool
Butter         bool
ShavingFoam    bool
Salt           bool
Flour          bool
HeavyCream     bool
Egg            bool
Olive          bool
Shampoo        bool
Sugar          bool
dtype: object

In [92]:
df_market.mean().sort_values(ascending= False).head()

Banana      0.448276
Cheese      0.443966
Bacon       0.431034
Hazelnut    0.420259
Honey       0.415948
dtype: float64

In [135]:
# Poda de conjuntos frecuentes, con soporte mínimo del 5% recomendado en datasets grandes
frequent_itemsets= apriori(df_market, min_support= 0.05, max_len= None, use_colnames= True)
print('Cantidad de reglas generadas:', len(frequent_itemsets))

Cantidad de reglas generadas: 3595


In [ ]:
# Extrae reglas de con confianza del 70%
positive_rules= association_rules(frequent_itemsets, metric= 'confidence', min_threshold= 0.7)
print('Cantidad de reglas generadas:', len(positive_rules))

Cantidad de reglas generadas: 264


In [158]:
positive_rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,"(Bacon, Onion)",(Banana),0.187500,0.448276,0.131466,0.701149,1.564103,1.0,0.047414,1.846154,0.443884,0.260684,0.458333,0.497209
1,"(Bread, Apple, Cheese)",(Honey),0.073276,0.415948,0.053879,0.735294,1.767754,1.0,0.023400,2.206418,0.468651,0.123762,0.546777,0.432414
2,"(Bread, Cheese, Meat)",(Honey),0.086207,0.415948,0.060345,0.700000,1.682902,1.0,0.024487,1.946839,0.444070,0.136585,0.486347,0.422539
3,"(Bread, Cheese, Carrot)",(Honey),0.112069,0.415948,0.079741,0.711538,1.710642,1.0,0.033126,2.024713,0.467856,0.177885,0.506103,0.451624
4,"(Bread, Honey, Carrot)",(Cheese),0.112069,0.443966,0.079741,0.711538,1.602689,1.0,0.029987,1.927586,0.423511,0.167421,0.481216,0.445575
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,"(ShavingFoam, Cheese, Butter, Meat)",(Bacon),0.066810,0.431034,0.053879,0.806452,1.870968,1.0,0.025082,2.939655,0.498845,0.121359,0.659824,0.465726
260,"(ShavingFoam, Butter, Bacon, Onion)",(Cheese),0.071121,0.443966,0.053879,0.757576,1.706384,1.0,0.022304,2.293642,0.445661,0.116822,0.564012,0.439467
261,"(ShavingFoam, Cheese, Butter, Onion)",(Bacon),0.071121,0.431034,0.053879,0.757576,1.757576,1.0,0.023224,2.346983,0.464037,0.120192,0.573921,0.441288
262,"(ShavingFoam, Cheese, Bacon, Onion)",(Butter),0.073276,0.375000,0.053879,0.735294,1.960784,1.0,0.026401,2.361111,0.528744,0.136612,0.576471,0.439486


In [159]:
# Primeras 10 reglas fuertes
positive_rules[['antecedents', 'consequents']].head(10)

,antecedents,consequents
0,"(Bacon, Onion)",(Banana)
1,"(Bread, Apple, Cheese)",(Honey)
2,"(Bread, Cheese, Meat)",(Honey)
3,"(Bread, Cheese, Carrot)",(Honey)
4,"(Bread, Honey, Carrot)",(Cheese)
5,"(Bread, Meat, Carrot)",(Honey)
6,"(Bread, Toothpaste, Bacon)",(ShavingFoam)
7,"(Bread, Apple, Bacon)",(Banana)
8,"(Bread, Cheese, Meat)",(Bacon)
9,"(Milk, Bread, Meat)",(Bacon)


In [160]:
# Si por ejemplo, se busca promocionar de forma cruzada el tocino, puede filtrarse en reglas donde está como consecuente
# para encontrar las mejores combinaciones
filtered_rules= positive_rules[positive_rules['consequents'].apply(lambda x: 'Bacon' in x)]

In [161]:
filtered_rules[['antecedents', 'consequents']]

,antecedents,consequents
8,"(Bread, Cheese, Meat)",(Bacon)
9,"(Milk, Bread, Meat)",(Bacon)
10,"(ShavingFoam, Bread, Meat)",(Bacon)
11,"(Sugar, Bread, Meat)",(Bacon)
12,"(ShavingFoam, Bread, Onion)",(Bacon)
...,...,...
253,"(Cheese, Butter, Banana, Egg)",(Bacon)
254,"(ShavingFoam, Hazelnut, Cheese, Butter)",(Bacon)
257,"(Hazelnut, Cheese, Butter, Egg)",(Bacon)
259,"(ShavingFoam, Cheese, Butter, Meat)",(Bacon)


In [162]:
# Ajuste de filtros para obtener reglas más específicas
filtered_rules_bacon= filtered_rules[(filtered_rules['antecedent support'] > 0.1) 
                                     & (filtered_rules['lift'] > 1.5) 
                                     & (filtered_rules['confidence'] > 0.7)]

In [165]:
# Tres primeras reglas de asociación
filtered_rules_bacon[['antecedents', 'consequents', 'antecedent support', 
                      'support', 'confidence', 'lift', 'conviction', 'zhangs_metric']].sort_values(by= 'confidence', ascending= False).head(3)

,antecedents,consequents,antecedent support,support,confidence,lift,conviction,zhangs_metric
76,"(Butter, Cheese, Banana)",(Bacon),0.114224,0.090517,0.792453,1.838491,2.741379,0.514888
112,"(ShavingFoam, Hazelnut, Butter)",(Bacon),0.107759,0.084052,0.780000,1.809600,2.586207,0.501425
125,"(Butter, Cheese, Egg)",(Bacon),0.105603,0.081897,0.775510,1.799184,2.534483,0.496639


<b>Conclusiones</b><br><br>
<b>Dataset Pizzería</b>:
- <b>Las reglas generadas no tienen fuerza suficiente para ser útiles en insights de negocio</b>:<br><br>
    - A pesar de que la pizza hawaiana y la pizza Thai de pollo son la combinación más vendida, la regla aparece en menos del 1% de las transacciones (Soporte), lo que no la hace representativa.
    - Baja predicción <b>antecedente -> consecuente</b> (Confianza al 5%).
    - Independencia y asociación negativa entre <b>antecedente -> consecuente</b> (Convicción = 0, Zhang negativo, Lift < 1).
<br><br>

<b>Dataset Mercado</b>:
- <b>Las reglas generadas sí tienen fuerza suficiente para ser útiles</b>:<br><br>
    - Al filtrar combinaciones donde aparece el tocino, la reglas aparecen en el 5% de las transacciones (Soporte), lo que es buena proporción en sets grandes.
    - Existe predicción <b>antecedente -> consecuente</b> (Confianza al 70%).
    - Dependencia y asociación positiva entre <b>antecedente -> consecuente</b> (Convicción > 1, Zhang > 0).
    - De acuerdo a los resultados, para promover las ventas de tocino, puede recurrirse a productos entre los cuales están la <b>mantequilla, queso, huevos, avellanas, entre otros.</b>
